In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f: \mathbb{R} \to \mathbb{R}, \quad y = \text{ReLU}(x) = \max(0, x) $$

$$ f:\mathbb{R^n} \to \mathbb{R^n}, \quad \mathbf{y} = \text{ReLU}(\mathbf{x}) = (f(x_1), f(x_2), \dots, f(x_n)) $$

$$ 
\frac{\partial f}{\partial x_i} = 
\begin{dcases}
    1 & x > 0 \\ 
    0 & x = 0 \\
    0 & x < 0
\end{dcases}
$$

$$ df = J_f(\mathbf{x}) \cdot d\mathbf{x} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{df}, df \right \rangle =
\left \langle \frac{dF}{d\mathbf{x}}, d\mathbf{x} \right \rangle
$$

$$
\left \langle \frac{dF}{df}, J_f(\mathbf{x}) \cdot d\mathbf{x} \right \rangle =
\left \langle \frac{dF}{d\mathbf{x}}, d\mathbf{x} \right \rangle \implies
$$

$$ \frac{dF}{d\mathbf{x}} = J_f(\mathbf{x})^\top \, \frac{dF}{df} $$

$$ \frac{dF}{d\mathbf{x}} = \frac{df}{d\mathbf{x}} \odot \frac{dF}{df} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def relu(x: tr.Tensor) -> tr.Tensor:
    return tr.maximum(tr.zeros_like(x), x)


class ReLUFunction(autograd.Function):
    """Custom autograd function for the `ReLU`."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        y = relu(x)
        return y

    @staticmethod
    def backward(ctx, dF_df):
        (x,) = ctx.saved_tensors

        df_dx = tr.ones_like(dF_df)
        df_dx[x < 0] = 0
        df_dx[x == 0] = 0

        df_dx = df_dx * dF_df
        
        return (df_dx, )
    
    
class ReLU(nn.Module):
    """Custom module for the `ReLU` function."""
    
    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return ReLUFunction.apply(x)

In [ ]:
def test_gradcheck(samples):
    def fn(x):
        return ReLUFunction.apply(x)

    z = tr.randn(samples, 1, dtype=tr.float64, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0, dtype=tr.float64),
        tr.tensor(+1.0, dtype=tr.float64)
    )

    assert autograd.gradcheck(fn, (z, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    z = tr.randn(samples, 1, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0),    
        tr.tensor(+1.0)
    ).reshape(samples, 1)

    z1 = z.clone().detach().requires_grad_(True)
    y1 = ReLU()(z1).sum()
    y1.backward()

    z2 = z.clone().detach().requires_grad_(True)
    y2 = nn.ReLU()(z2).sum()
    y2.backward()

    assert y1.item() == approx(y2.item())
    assert z1.grad == approx(z2.grad)


def test_gradcheck(samples = 10) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return ReLUFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)

if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)